# AMI aberration basis inspector

Builds five candidate aberration bases **exactly as the pipeline would see them** (per-hole,
cropped to the `small_npix` window, evaluated on the *oversized* hexagon `f2f * oversize`), and lets you
inspect the first `N_MODES` vectors of each, on a single hole and assembled on the full AMI pupil.

| name | what it is |
|---|---|
| `zernike` | plain Zernikes on the hole's coordinates, cropped (`calc_basis(..., polike=False)`) |
| `polike` | polygon-mapped Zernikes (`dlu.polike_basis`, `calc_basis(..., polike=True)`) |
| `fourier` | raw separable Fourier dictionary, first N by spatial frequency, masked and unit-RMS normalised |
| `zernike_orth` | the `zernike` modes Gram-Schmidt / QR orthogonalised over `ORTH_ON` (Noll order kept, so mode *k* = Zernike *k* minus its projection onto modes < *k*) |
| `fourier_orth` | SVD-orthogonalised Fourier dictionary over `ORTH_ON` (with `ORTH_ON="oversized hexagon"` this is exactly the pipeline's `orthogonalise_fourier_basis`) |

Everything is scaled to RMS ~ 1 inside the oversized hexagon (the Zernike convention the learning rates were tuned for).
**Note the pipeline evaluates the basis on the oversized hexagon (`oversize=1.1`) but light only passes through the
`f2f=0.80` hexagon**, so orthogonality over the oversized hexagon does not imply orthogonality over the real aperture.
The diagnostics below let you choose which mask to measure on, and `ORTH_ON` chooses which region the two *orthogonalised* bases are made orthogonal over
(default: the real aperture; set it to `"oversized hexagon"` to reproduce what the pipeline builds today).

In [ ]:
import math, warnings
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as np
from jax import vmap
import numpy as onp
import dLux.utils as dlu
import matplotlib.pyplot as plt
from amigo import optical_models as om

warnings.filterwarnings("ignore")
plt.rcParams["image.origin"] = "lower"

# ---- knobs -----------------------------------------------------------------
N_MODES = 28            # first n modes of every basis
FOURIER_DICT_N = 13     # per-axis Fourier dictionary size (13 -> 169 modes), as in the pipeline
F2F, OVERSIZE = 0.80, 1.1
ORTH_ON = "true aperture"     # or "oversized hexagon" (= what the pipeline's fourier=True builds today)
DIAMETER, NPIX, SMALL = 6.603464, 1024, 180

## Geometry (identical to `StaticApertureMask.__init__`)

In [ ]:
coords = dlu.pixel_coords(NPIX, DIAMETER)
holes = om.get_initial_holes(DIAMETER, NPIX)
hole_coords = vmap(dlu.translate_coords, (None, 0))(coords, holes)
corners = om.calc_hole_corners(coords, holes, SMALL)
small = om.crop_hole_coords(hole_coords, corners, SMALL)     # (n_holes, 2, SMALL, SMALL)
pix = DIAMETER / NPIX
n_holes = small.shape[0]
f2f_ab = F2F * OVERSIZE                                       # what the aberration basis is built on

hexmask = lambda f: vmap(lambda c: dlu.soft_reg_polygon(c, f / np.sqrt(3), 6, pix))(small)
mask_true = onp.asarray(hexmask(F2F))          # real transmitting aperture
mask_over = onp.asarray(hexmask(f2f_ab))       # oversized hexagon the bases live on
transmission = om.calc_mask(hole_coords, F2F, pix)            # full-pupil transmission
n_valid_over = (mask_over > 0.5).sum((-1, -2))
print(f"{n_holes} holes, window {SMALL}px, pixel {pix*1e3:.2f} mm, "
      f"valid px: true {(mask_true>0.5).sum((-1,-2))[0]}, oversized {n_valid_over[0]}")

## Build the five bases  (`bases[name]` has shape `(n_holes, N_MODES, SMALL, SMALL)`)

In [ ]:
# radial orders needed to cover N_MODES Noll modes
R_ORD = next(r for r in range(1, 30) if r * (r + 1) // 2 >= N_MODES)
unit_rms = lambda B, m: B / onp.sqrt(((B * m[:, None]) ** 2).sum((-1, -2), keepdims=True) / (m > 0.5).sum((-1, -2))[:, None, None, None])

bases = {}
bases["zernike"] = onp.asarray(om.calc_basis(small, f2f_ab, R_ORD, polike=False))[:, :N_MODES]
bases["polike"] = onp.asarray(om.calc_basis(small, f2f_ab, R_ORD, polike=True))[:, :N_MODES]

# raw Fourier: dictionary is independent of hole position, so build once. Order by spatial frequency.
fd = onp.asarray(om.calc_fourier_dict(small[0], f2f_ab, FOURIER_DICT_N))          # (169, S, S)
ii, jj = onp.divmod(onp.arange(FOURIER_DICT_N ** 2), FOURIER_DICT_N)
fx, fy = (ii + 1) // 2, (jj + 1) // 2
perm = onp.lexsort((jj, ii, fx ** 2 + fy ** 2))[:N_MODES]
fourier_labels = [f"kx={fx[p]}{'c' if ii[p]%2 else 's'} ky={fy[p]}{'c' if jj[p]%2 else 's'}" for p in perm]
bases["fourier"] = unit_rms(onp.broadcast_to(fd[perm], (n_holes, N_MODES, SMALL, SMALL)) * mask_over[:, None], mask_over)

def gram_schmidt(B, m):
    # QR over the aperture: columns = modes. R is the exact old->new coefficient map (c_new = R c_old).
    out, Rs = [], []
    for h in range(B.shape[0]):
        F = (B[h] * m[h]).reshape(B.shape[1], -1).T
        Q, R = np.linalg.qr(F)
        s = np.sign(np.diag(R)); s = np.where(s == 0, 1, s)
        Q = Q * s
        out.append(onp.asarray(Q.T.reshape(B.shape[1], *B.shape[2:])) * onp.sqrt((m[h] > 0.5).sum()))
        Rs.append(onp.asarray(R * s[:, None]))
    return onp.stack(out), onp.stack(Rs)

orth_mask = mask_true if ORTH_ON == "true aperture" else mask_over
bases["zernike_orth"], R_zorth = gram_schmidt(bases["zernike"], orth_mask)

def svd_orth(m):
    F = (fd[None] * m[None, None]).reshape(fd.shape[0], -1).T
    U, s_, Vt = np.linalg.svd(F, full_matrices=False)
    return onp.asarray(U[:, :N_MODES].T.reshape(N_MODES, SMALL, SMALL)) * onp.sqrt((m > 0.5).sum())

bases["fourier_orth"] = onp.stack([svd_orth(orth_mask[h]) for h in range(n_holes)])
if ORTH_ON == "oversized hexagon":
    ref = onp.asarray(om.orthogonalise_fourier_basis(small[0], f2f_ab, FOURIER_DICT_N, N_MODES))
    # individual SVD modes are only defined up to sign / rotation within (near-)degenerate singular-value
    # groups (the hexagon is 6-fold symmetric), so compare the *spanned subspace*, not mode-by-mode.
    v = orth_mask[0] > 0.5
    A, Bm = bases["fourier_orth"][0][:, v].T, ref[:, v].T
    resid = Bm - A @ onp.linalg.lstsq(A, Bm, rcond=None)[0]
    print("pipeline orthogonalise_fourier_basis vs this cell (hole 0): subspace residual", onp.abs(resid).max(), "(basis values ~1)")
NAMES = list(bases)
print({k: v.shape for k, v in bases.items()})

## Helpers
`MASKS` picks the region diagnostics/plots are measured on: the real 0.80 aperture or the oversized hexagon the bases are built on.

In [ ]:
MASKS = {"true aperture (f2f=0.80)": mask_true, "oversized hexagon (x1.1)": mask_over}

def masked(B, m):
    return onp.where(m > 0.5, B, onp.nan)

def corr_matrix(A, B, m):
    # normalised inner product over the mask: (N,N) matrix of <a_i, b_j> / (|a_i||b_j|)
    v = m > 0.5
    a, b = A[:, v], B[:, v]
    G = a @ b.T
    return G / onp.outer(onp.sqrt((a * a).sum(1)), onp.sqrt((b * b).sum(1)))

def summarise(name, hole, m):
    C = corr_matrix(bases[name][hole], bases[name][hole], m[hole])
    off = onp.abs(C - onp.eye(len(C)))
    w = onp.linalg.eigvalsh(C)
    return off.max(), w.max() / max(w.min(), 1e-15)

## 1. Orthogonality / conditioning diagnostics

In [ ]:
def diagnostics(hole=0, mask_name=list(MASKS)[0]):
    m = MASKS[mask_name]
    fig, ax = plt.subplots(1, len(NAMES), figsize=(4.2 * len(NAMES), 4.2))
    for a, name in zip(ax, NAMES):
        C = corr_matrix(bases[name][hole], bases[name][hole], m[hole])
        off, cond = summarise(name, hole, m)
        im = a.imshow(C, vmin=-1, vmax=1, cmap="RdBu_r", origin="upper")
        a.set_title(f"{name}\nmax|offdiag|={off:.3f}  cond={cond:.1f}", fontsize=9); a.set(xlabel="mode", ylabel="mode")
    fig.colorbar(im, ax=ax, shrink=0.8, label="normalised inner product")
    fig.suptitle(f"hole {hole}, measured over: {mask_name}", y=1.02)
    plt.show()

for mn in MASKS:
    diagnostics(0, mn)

In [ ]:
# per-mode RMS over the chosen mask, and linear-fit (piston+tip+tilt) power of each mode
def per_mode_stats(hole=0, mask_name=list(MASKS)[0]):
    m = MASKS[mask_name][hole] > 0.5
    x, y = [onp.asarray(c)[m] for c in small[hole]]
    P = onp.stack([onp.ones_like(x), x, y], 1)
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
    for name in NAMES:
        B = bases[name][hole][:, m]
        ax[0].plot(onp.sqrt((B ** 2).mean(1)), "o-", ms=3, label=name)
        coef = onp.linalg.lstsq(P, B.T, rcond=None)[0]
        r2 = ((P @ coef) ** 2).sum(0) / (B ** 2).sum(1)
        ax[1].plot(r2, "o-", ms=3, label=name)
    ax[0].set(title="RMS per mode", xlabel="mode", ylabel="RMS"); ax[0].legend()
    ax[1].set(title="power in piston+tip+tilt (leakage into low order)", xlabel="mode", ylabel="fraction"); ax[1].set_yscale("symlog", linthresh=1e-3)
    fig.suptitle(f"hole {hole}, over: {mask_name}", y=1.02); plt.show()

per_mode_stats(0, list(MASKS)[0])

## 2. Are `polike` and `zernike_orth` the same thing?
Cross-correlation of the two bases (rows: `polike`, cols: `zernike_orth`). A near-diagonal matrix means `polike` ~ Gram-Schmidt Zernike; off-diagonal
structure shows where they differ. Any pair of bases can be compared.

In [ ]:
def cross(a="polike", b="zernike_orth", hole=0, mask_name=list(MASKS)[0]):
    C = corr_matrix(bases[a][hole], bases[b][hole], MASKS[mask_name][hole])
    plt.figure(figsize=(5.5, 4.8)); plt.imshow(C, vmin=-1, vmax=1, cmap="RdBu_r", origin="upper"); plt.colorbar(label="normalised inner product")
    plt.gca().set(title=f"<{a}, {b}> hole {hole}\n({mask_name})", xlabel=b, ylabel=a); plt.show()

cross("polike", "zernike_orth")
cross("zernike", "zernike_orth")

## 3. polike vs the real hexagon
`polike` assumes a particular polygon orientation (vertices at 30/90/150 deg). If the real aperture is rotated relative to that, polike
is no longer orthogonal and shows edge artefacts. IoU of polike's implied hexagon vs the actual mask, at 0 and 30 deg rotation:

In [ ]:
def polike_hex(c, rot=0.0):
    x, y = c[0], c[1]
    th = np.arctan2(y, x) + rot
    al = np.pi / 6
    u = th + al - np.floor((th + 2 * al) / (2 * al)) * 2 * al
    r_al = np.cos(al) / np.cos(u)
    return (np.hypot(x, y) / (f2f_ab / np.sqrt(3))) <= r_al

for rot_deg in (0, 30):
    ph = onp.asarray(vmap(lambda c: polike_hex(c, onp.deg2rad(rot_deg)))(small))
    real = mask_over > 0.5
    iou = (ph & real).sum((-1, -2)) / (ph | real).sum((-1, -2))
    print(f"rot {rot_deg:2d} deg: IoU polike-hexagon vs real oversized mask, holes: min {iou.min():.4f} mean {iou.mean():.4f}")

fig, ax = plt.subplots(1, 2, figsize=(9, 4.2))
ph0 = onp.asarray(polike_hex(small[0])); ax[0].imshow(ph0 * 1.0 + 2.0 * (mask_over[0] > 0.5), cmap="viridis")
ax[0].set(title="polike hexagon (1) + real mask (2); overlap = 3")
ph30 = onp.asarray(polike_hex(small[0], onp.deg2rad(30))); ax[1].imshow(ph30 * 1.0 + 2.0 * (mask_over[0] > 0.5), cmap="viridis")
ax[1].set(title="same, polike rotated 30 deg"); plt.show()

## 4. Single-hole mode viewer

In [ ]:
def show_modes(name="zernike", hole=0, n=12, mask_name=list(MASKS)[1], ncols=6, outline=True):
    B = masked(bases[name][hole][:n], MASKS[mask_name][hole])
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.6 * ncols, 2.6 * nrows), squeeze=False)
    for k, a in enumerate(axes.ravel()):
        a.axis("off")
        if k >= n: continue
        v = onp.nanmax(onp.abs(B[k]))
        a.imshow(B[k], cmap="RdBu_r", vmin=-v, vmax=v)
        if outline:
            a.contour(mask_true[hole], [0.5], colors="k", linewidths=0.6)
            a.contour(mask_over[hole], [0.5], colors="0.5", linewidths=0.6, linestyles=":")
        lab = f"{k}" + (f"  {fourier_labels[k]}" if name == "fourier" else "")
        a.set_title(f"{lab}  (max {v:.2f})", fontsize=8)
    fig.suptitle(f"{name}, hole {hole}, first {n} modes (solid: real aperture, dotted: oversized hexagon)", y=1.0)
    plt.tight_layout(); plt.show()

show_modes("zernike", n=N_MODES)

## 5. Assembled on the AMI pupil

In [ ]:
def pupil_map(name, k, mask_name=list(MASKS)[0]):
    B = bases[name][:, k] * (MASKS[mask_name] > 0.5)            # (n_holes, S, S), zero outside the chosen aperture
    return onp.asarray(om.fill(np.asarray(B), corners, NPIX))

def show_pupils(k=3, names=NAMES, mask_name=list(MASKS)[0], crop=True):
    maps = {n: pupil_map(n, k, mask_name) for n in names}
    sup = onp.abs(onp.asarray(transmission)) > 0
    ys, xs = onp.where(sup)
    sl = (slice(ys.min() - 10, ys.max() + 10), slice(xs.min() - 10, xs.max() + 10)) if crop else (slice(None), slice(None))
    fig, ax = plt.subplots(1, len(names), figsize=(4.4 * len(names), 4.4))
    for a, n in zip(onp.atleast_1d(ax), names):
        im = maps[n][sl]; v = onp.abs(im).max() or 1
        a.imshow(onp.where(im == 0, onp.nan, im), cmap="RdBu_r", vmin=-v, vmax=v)
        a.set(title=f"{n}  mode {k}"); a.axis("off")
    fig.suptitle(f"mode {k} on the pupil (each hole has its own coefficient; masked to: {mask_name})", y=1.0)
    plt.tight_layout(); plt.show()

show_pupils(3)

## 6. Interactive
Sliders for basis / mode / hole / mask. (Requires a live Jupyter kernel; the static cells above cover the same views.)

In [ ]:
import ipywidgets as w
from IPython.display import display

basis_dd = w.Dropdown(options=NAMES, value="zernike", description="basis")
mask_dd = w.Dropdown(options=list(MASKS), value=list(MASKS)[0], description="mask")
hole_s = w.IntSlider(0, 0, n_holes - 1, description="hole")
mode_s = w.IntSlider(3, 0, N_MODES - 1, description="mode k")
n_s = w.IntSlider(12, 1, N_MODES, description="first n")

w.interact(lambda name, hole, n, mask_name: show_modes(name, hole, n, mask_name), name=basis_dd, hole=hole_s, n=n_s, mask_name=mask_dd)
w.interact(lambda k, mask_name: show_pupils(k, NAMES, mask_name), k=mode_s, mask_name=mask_dd)
w.interact(lambda hole, mask_name: diagnostics(hole, mask_name), hole=hole_s, mask_name=mask_dd)

## 7. Converting existing Zernike coefficients into the orthogonalised basis
`R_zorth[h]` is the upper-triangular matrix with `Z = Q R` (modes as columns), so a wavefront `Z c_old` equals `Q (R c_old)`.
The exact new coefficients for a fitted state are `c_new = R c_old`, which is what lets a converged Zernike checkpoint be loaded into `zernike_orth`
(exact for the first N modes because R is upper-triangular, so truncation commutes with the change of basis). Wavefronts are compared over `ORTH_ON`.

In [ ]:
c_old = onp.random.default_rng(0).normal(size=N_MODES)          # a random Zernike wavefront
h = 0
nv = (orth_mask[h] > 0.5).sum()
wf_old = onp.tensordot(c_old, bases["zernike"][h], 1) * orth_mask[h]
c_new = R_zorth[h] @ c_old / onp.sqrt(nv)                        # basis_orth = Q*sqrt(nv), masked Z = Q R  ->  c_new = R c_old / sqrt(nv)
wf_new = onp.tensordot(c_new, bases["zernike_orth"][h], 1)
print("max |wavefront difference| after conversion:", onp.abs(wf_old - wf_new).max(), " (wavefront max:", onp.abs(wf_old).max(), ")")